# Task 04 — Improving Performance

**Agent:** `<claude / codex / antigravity>`  
**Date:** `<YYYY-MM-DD>`  
**Inputs:** `agents/<agent>/task_01/outputs/ingested.csv` + `agents/<agent>/task_03/outputs/`

---

**Objective:** Beat the baseline via feature engineering, tuning, or a different model. Show baseline vs improved side by side.

> Before committing: **Kernel → Restart & Clear Output**

In [ ]:
# ── Imports & seed ──────────────────────────────────────────────────────────
import random
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INGESTED_PATH   = Path("../task_01/outputs/ingested.csv")
BASELINE_PATH   = Path("../task_03/outputs/baseline_results.csv")
OUTPUT_DIR      = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data & Baseline

In [ ]:
df = pd.read_csv(INGESTED_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)

# TODO: set TARGET to the name of your target column
TARGET = "<target_column>"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print("Baseline results:")
print(baseline_df.to_string(index=False))

## 2. Improvement Strategy

> Choose at least one: feature engineering / hyperparameter tuning / different model.  
> All tuning must use cross-validation on training data — never the test set.

In [ ]:
# TODO: feature engineering — add new columns, transform existing ones
# Example:
# X_train["log_area"] = np.log1p(X_train["GrLivArea"])
# X_test["log_area"]  = np.log1p(X_test["GrLivArea"])  # same transform, no fit

# TODO: define your improved pipeline
from sklearn.ensemble import RandomForestClassifier  # replace as appropriate

improved_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  RandomForestClassifier(n_estimators=200, random_state=SEED))
])

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()

## 3. Validate via Cross-Validation (train only)

In [ ]:
cv_scores = cross_val_score(
    improved_pipeline, X_train[numeric_cols], y_train,
    cv=5, scoring="f1_weighted"
)
print(f"CV F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 4. Train on Full Train Set & Evaluate on Test

In [ ]:
improved_pipeline.fit(X_train[numeric_cols], y_train)
y_pred  = improved_pipeline.predict(X_test[numeric_cols])
y_proba = improved_pipeline.predict_proba(X_test[numeric_cols])[:, 1] \
          if hasattr(improved_pipeline.named_steps["model"], "predict_proba") else None

improved_results = {
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "f1":       round(f1_score(y_test, y_pred, average="weighted"), 4),
}
if y_proba is not None:
    improved_results["auc"] = round(roc_auc_score(y_test, y_proba), 4)

for k, v in improved_results.items():
    print(f"  {k:12s}: {v}")

## 5. Compare Baseline vs Improved

In [ ]:
baseline_dict = dict(zip(baseline_df["metric_name"], baseline_df["value"]))

comparison_rows = []
for metric, improved_val in improved_results.items():
    baseline_val = baseline_dict.get(metric, float("nan"))
    comparison_rows.append({
        "metric_name":    metric,
        "baseline_value": baseline_val,
        "improved_value": improved_val,
        "delta":          round(improved_val - float(baseline_val), 4)
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

## 6. Save Outputs

In [ ]:
# improved_results.csv
comparison_df.to_csv(OUTPUT_DIR / "improved_results.csv", index=False)

# improved_model.pkl
with open(OUTPUT_DIR / "improved_model.pkl", "wb") as f:
    pickle.dump(improved_pipeline, f)

# improvement_report.md
report = f"""# Improvement Report

**SEED:** {SEED}  
**Improved model:** {type(improved_pipeline.named_steps['model']).__name__}  

## What I tried
<!-- TODO: describe every technique attempted -->
1. 

## Results
{comparison_df.to_markdown(index=False)}

## Leakage check
<!-- TODO: confirm no leakage was introduced -->
- All new preprocessing steps fitted on training data only.
- Hyperparameters selected via cross-validation, not the test set.

## Why it worked (or didn't)
<!-- TODO: interpretation -->
"""
(OUTPUT_DIR / "improvement_report.md").write_text(report)
print("Saved: improved_results.csv, improved_model.pkl, improvement_report.md")